
# FineWeb Filtering + Evaluation Notebook

This notebook is used to complete the following tasks:

1. Filter the parquet data according to the rules you specified
2. Export CSV samples for cleaned / dropped / borderline data
3. Output basic filtering results
4. Output short summaries of findings
5. Compare evaluation metrics before vs. after cleaning:

   * language_score distribution
   * token_count distribution
   * text length distribution
   * domain distribution
   * sample inspection (dropped samples, borderline samples)
   * dirty data signals (boilerplate, HTML residue)
   * number of hits for each individual rule



In [13]:

from pathlib import Path
from urllib.parse import urlparse
from collections import Counter
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import random
import re
import os
import json

pd.set_option("display.max_colwidth", 200)

# ========= **User Configuration Section** =========
INPUT_PATH = "data/fineweb/004_00018.parquet"      # either a single parquet file or a directory.
OUTPUT_DIR = "filter_outputs"
BATCH_SIZE = 2000
RANDOM_SEED = 42

# Number of samples to export**
CLEANED_SAMPLE_N = 100
DROPPED_SAMPLE_N = 100
BORDERLINE_SAMPLE_N = 100

# Rule Thresholds**
MIN_TOKEN_COUNT = 50
MIN_TEXT_LENGTH = 200
MIN_LANGUAGE_SCORE = 0.85
MAX_TOKEN_COUNT = 10000

# Borderline Sample Range**
BORDERLINE_LANG_LOW = 0.85
BORDERLINE_LANG_HIGH = 0.90

# URL blocklist path keywords
BLOCKED_URL_PATTERNS = [
    "/login",
    "/signin",
    "/signup",
    "/register",
    "/cart",
    "/checkout",
    "/account",
]

# boilerplate signals
BOILERPLATE_PATTERNS = [
    "add to cart",
    "buy now",
    "subscribe",
    "sign up",
    "free shipping",
    "accept cookies",
    "privacy policy",
    "terms of service",
    "related products",
    "customer reviews",
    "out of stock",
]

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("INPUT_PATH =", os.path.abspath(INPUT_PATH))
print("OUTPUT_DIR =", os.path.abspath(OUTPUT_DIR))


INPUT_PATH = /home/jovyan/data/fineweb/004_00018.parquet
OUTPUT_DIR = /home/jovyan/filter_outputs


In [2]:

def resolve_parquet_files(input_path):
    p = Path(input_path)
    if p.is_file():
        return [p]
    if p.is_dir():
        files = sorted(p.glob("*.parquet"))
        if files:
            return files
        raise FileNotFoundError(f"No parquet files found in directory: {p}")
    raise FileNotFoundError(f"Path does not exist: {p}")

def get_domain(url):
    try:
        domain = urlparse(str(url)).netloc.lower().strip()
        if domain.startswith("www."):
            domain = domain[4:]
        return domain
    except Exception:
        return None

def describe_series(values):
    s = pd.Series(values)
    if len(s) == 0:
        return {"count": 0}
    s = pd.to_numeric(s, errors="coerce").dropna()
    if len(s) == 0:
        return {"count": 0}
    return {
        "count": int(s.count()),
        "mean": float(s.mean()),
        "median": float(s.median()),
        "min": float(s.min()),
        "p25": float(s.quantile(0.25)),
        "p75": float(s.quantile(0.75)),
        "max": float(s.max()),
    }

def contains_blocked_path(url):
    if pd.isna(url):
        return False
    u = str(url).lower()
    return any(pat in u for pat in BLOCKED_URL_PATTERNS)

def boilerplate_hit_count(text):
    if pd.isna(text):
        return 0
    t = str(text).lower()
    return sum(pat in t for pat in BOILERPLATE_PATTERNS)

def has_html_residue(text):
    if pd.isna(text):
        return False
    t = str(text).lower()
    html_patterns = [
        "<div", "<span", "<script", "<style", "<a ", "</a>", "href=", "class=", "&nbsp;", "&amp;"
    ]
    return any(p in t for p in html_patterns)

def text_length(text):
    if pd.isna(text):
        return np.nan
    return len(str(text))

def compute_rule_flags(df):
    out = df.copy()

    out["text_is_null"] = out["text"].isna()
    out["url_is_null"] = out["url"].isna()
    out["language_score_is_null"] = out["language_score"].isna()
    out["token_count_is_null"] = out["token_count"].isna()

    out["token_count_num"] = pd.to_numeric(out["token_count"], errors="coerce")
    out["language_score_num"] = pd.to_numeric(out["language_score"], errors="coerce")
    out["text_length"] = out["text"].map(text_length)
    out["domain"] = out["url"].map(get_domain)

    out["rule_token_lt_50"] = out["token_count_num"] < MIN_TOKEN_COUNT
    out["rule_text_len_lt_200"] = out["text_length"] < MIN_TEXT_LENGTH
    out["rule_lang_lt_085"] = out["language_score_num"] < MIN_LANGUAGE_SCORE
    out["rule_token_gt_10000"] = out["token_count_num"] > MAX_TOKEN_COUNT
    out["rule_url_blocked"] = out["url"].map(contains_blocked_path)

    out["boilerplate_hits"] = out["text"].map(boilerplate_hit_count)
    out["has_boilerplate"] = out["boilerplate_hits"] > 0
    out["has_html_residue"] = out["text"].map(has_html_residue)

    rule_cols = [
        "text_is_null",
        "url_is_null",
        "language_score_is_null",
        "token_count_is_null",
        "rule_token_lt_50",
        "rule_text_len_lt_200",
        "rule_lang_lt_085",
        "rule_token_gt_10000",
        "rule_url_blocked",
    ]
    out["drop_reason_count"] = out[rule_cols].sum(axis=1)
    out["is_dropped"] = out["drop_reason_count"] > 0

    out["is_borderline_lang"] = (
        out["language_score_num"].notna()
        & (out["language_score_num"] >= BORDERLINE_LANG_LOW)
        & (out["language_score_num"] < BORDERLINE_LANG_HIGH)
    )

    return out

PARQUET_FILES = resolve_parquet_files(INPUT_PATH)
print("Found parquet files:", len(PARQUET_FILES))
for p in PARQUET_FILES[:10]:
    print(" -", p)
if len(PARQUET_FILES) > 10:
    print(" ...")


Found parquet files: 1
 - data/fineweb/004_00018.parquet



## 1. Check the basic information of the input data.


In [3]:

first_file = PARQUET_FILES[0]
pf = pq.ParquetFile(first_file)

print("=== Basic Info ===")
print("File path:", os.path.abspath(first_file))
print(f"File size: {os.path.getsize(first_file) / (1024 ** 2):.2f} MB")
print("Row groups:", pf.num_row_groups)
print("Number of rows:", pf.metadata.num_rows)
print("Number of columns:", pf.metadata.num_columns)

print("\n=== Schema ===")
schema = pf.schema_arrow
for field in schema:
    print(f"{field.name} --> {field.type}")

dataset_preview = ds.dataset(first_file, format="parquet")
sample_df = dataset_preview.head(10).to_pandas()

print("\n=== Sample Rows ===")
display(sample_df)


=== Basic Info ===
File path: /home/jovyan/data/fineweb/004_00018.parquet
File size: 378.87 MB
Row groups: 173
Number of rows: 172447
Number of columns: 9

=== Schema ===
text --> string
id --> string
dump --> string
url --> string
date --> string
file_path --> string
language --> string
language_score --> double
token_count --> int64

=== Sample Rows ===


,text,id,dump,url,date,file_path,language,language_score,token_count
0,"Welcome to Harrisonburg TownSquare.\nTownSquare brings together all your favorite local shops and makers in one cozy corner of the internet, so you can support your neighbors and find something tr...",<urn:uuid:189514cf-784a-4e9f-9e8e-f350c5a41adf>,CC-MAIN-2025-26,https://www.townsquareapp.co/,2025-06-16T16:22:40Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/segments/1749709481416.36/warc/CC-MAIN-20250616145137-20250616175137-00968.warc.gz,en,0.899284,203
1,"The Art of Effective Board Reporting\nSep 5, 2024\nBoard reporting is a cornerstone of corporate governance, serving as the primary channel through which the board of directors gains insight into ...",<urn:uuid:9d2f701d-48f5-418f-a8e2-d5b2d51aad36>,CC-MAIN-2025-26,https://www.traact.com/blog/the-art-of-effective-board-reporting,2025-06-16T16:45:14Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/segments/1749709481416.36/warc/CC-MAIN-20250616145137-20250616175137-00968.warc.gz,en,0.942698,2717
2,Belinda's journey and transformation is a real triumph over so much personal stress and pressure. I have a huge amount of respect and admiration for Belinda who has kept going despite all the setb...,<urn:uuid:ccb16b4e-f301-480c-9231-33f6e823eabe>,CC-MAIN-2025-26,https://www.tracydixonmindandbodyfit.com/post/2019/08/28/i-wanted-to-be-around-for-my-daughter-when-she-grew-up-i-knew-i-needed-to-make-changes,2025-06-16T15:44:36Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/segments/1749709481416.36/warc/CC-MAIN-20250616145137-20250616175137-00968.warc.gz,en,0.987144,555
3,Many people purchase plain shirts or other clothing items so they can later add embroidery and designs to them. They want to personalize their own items to make them more their own. Embroidering i...,<urn:uuid:42451264-d511-45ac-8afb-d27558b16a33>,CC-MAIN-2025-26,https://www.transportdirectory.org/items-that-allow-the-use-of-embroidery-in-olathe/,2025-06-16T16:20:43Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/segments/1749709481416.36/warc/CC-MAIN-20250616145137-20250616175137-00968.warc.gz,en,0.976207,469
4,"Beijing Itinerary 5 Days: What to See & How to Plan\nDay 1: Tiananmen Square, Forbidden City, Jingshan Park, Beihai Park, Wangfujing\nYou may go first to Tiananmen Square and Forbidden City, for t...",<urn:uuid:b3df7fbd-18f6-4bab-b074-969a3f6425c5>,CC-MAIN-2025-26,https://www.travelchinaguide.com/package/5-days-in-beijing.htm,2025-06-16T16:09:17Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/segments/1749709481416.36/warc/CC-MAIN-20250616145137-20250616175137-00968.warc.gz,en,0.946699,1496
5,"The enchantment of Hong Kong is undeniable: city wonders, a glimmering harbour, and upbeat energy fill the city. Just over 700 kilometres across the East China Sea in Taiwan lies an otherworldly l...",<urn:uuid:a3251d97-2ae9-4371-bfdd-9d2cc01458c9>,CC-MAIN-2025-26,https://www.travelfortravellers.com/2017/08/5-reasons-visit-hong-kong-taiwan/,2025-06-16T16:56:10Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/segments/1749709481416.36/warc/CC-MAIN-20250616145137-20250616175137-00968.warc.gz,en,0.907712,1440
6,"Yahja: ""Jan the Survivor""\nJanuary 5, 2007\nDear Family and Friends,\nHat Co...Sa Qax,\nYes, I am a survivor!\nSo far on this trip to Centro America I have survived the four hour angry-bull-ride d...",<urn:uuid:41493774-e601-412c-9fde-d2f527278162>,CC-MAIN-2025-26,https://www.travelwithjan.com/america2007yahja,2025-06-16T16:16:52Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/segments/1749709481416.36/warc/CC-MAIN-20250616145137-20250616175137-00968.warc.gz,en,0.903129,768
7,"Having a baby is often described as an exciting time filled with love, joy, and new experiences. However, the reality for many new parents is more complex. Yes, there are beautiful moments, but th...",<urn:uuid:cb615e69-29e8-4885-b1d1-11b350af86c0>,CC-MAIN-2025-26,https://www.treatmyocd.com/blog/postpartum-intrusive-thoughts,2025-06-16T16:54:53Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26


## 2. Full Scan and Rule-Based Filtering

This section will:
- read the parquet file batch by batch
- determine whether each rule is triggered
- split the data into cleaned / dropped / borderline
- compute before / after statistics
- export CSV samples

In [5]:

columns_to_read = ["text", "url", "language_score", "token_count"]

total_rows = 0

# before stats
before_token_values = []
before_lang_values = []
before_text_len_values = []
before_domain_counter = Counter()
before_null_counter = Counter()
before_boilerplate_docs = 0
before_html_docs = 0

# after stats
after_token_values = []
after_lang_values = []
after_text_len_values = []
after_domain_counter = Counter()
after_boilerplate_docs = 0
after_html_docs = 0

# rule hit counts
rule_hit_counter = Counter()

# sample collectors
cleaned_samples = []
dropped_samples = []
borderline_samples = []

def append_sample_rows(target_list, df_part, max_n):
    if len(target_list) >= max_n:
        return
    need = max_n - len(target_list)
    if len(df_part) == 0:
        return
    picked = df_part.head(need).copy()
    target_list.append(picked)

for parquet_path in PARQUET_FILES:
    print(f"Processing: {parquet_path}")
    dataset = ds.dataset(parquet_path, format="parquet")
    scanner = dataset.scanner(columns=columns_to_read, batch_size=BATCH_SIZE)

    for batch in scanner.to_batches():
        df = batch.to_pandas()
        total_rows += len(df)

        # before
        before_null_counter.update(df.isna().sum().to_dict())

        token_vals = pd.to_numeric(df["token_count"], errors="coerce").dropna()
        before_token_values.extend(token_vals.tolist())

        lang_vals = pd.to_numeric(df["language_score"], errors="coerce").dropna()
        before_lang_values.extend(lang_vals.tolist())

        text_lens = df["text"].dropna().astype(str).map(len)
        before_text_len_values.extend(text_lens.tolist())

        for url in df["url"].dropna():
            d = get_domain(url)
            if d:
                before_domain_counter[d] += 1

        before_boilerplate_docs += df["text"].map(lambda x: boilerplate_hit_count(x) > 0).sum()
        before_html_docs += df["text"].map(has_html_residue).sum()

        # flags
        flagged = compute_rule_flags(df)

        # rule hit counts
        rule_hit_counter["text is null"] += int(flagged["text_is_null"].sum())
        rule_hit_counter["url is null"] += int(flagged["url_is_null"].sum())
        rule_hit_counter["language_score is null"] += int(flagged["language_score_is_null"].sum())
        rule_hit_counter["token_count is null"] += int(flagged["token_count_is_null"].sum())
        rule_hit_counter["token_count < 50"] += int(flagged["rule_token_lt_50"].sum())
        rule_hit_counter["len(text) < 200"] += int(flagged["rule_text_len_lt_200"].sum())
        rule_hit_counter["language_score < 0.85"] += int(flagged["rule_lang_lt_085"].sum())
        rule_hit_counter["token_count > 10000"] += int(flagged["rule_token_gt_10000"].sum())
        rule_hit_counter["url path blocked"] += int(flagged["rule_url_blocked"].sum())

        kept = flagged[~flagged["is_dropped"]].copy()
        dropped = flagged[flagged["is_dropped"]].copy()
        borderline = flagged[flagged["is_borderline_lang"]].copy()

        # after
        after_token_values.extend(kept["token_count_num"].dropna().tolist())
        after_lang_values.extend(kept["language_score_num"].dropna().tolist())
        after_text_len_values.extend(kept["text_length"].dropna().tolist())

        for d in kept["domain"].dropna():
            after_domain_counter[d] += 1

        after_boilerplate_docs += int(kept["has_boilerplate"].sum())
        after_html_docs += int(kept["has_html_residue"].sum())

        # samples
        cleaned_cols = ["url", "domain", "language_score_num", "token_count_num", "text_length", "boilerplate_hits", "has_html_residue", "text"]
        dropped_cols = cleaned_cols + [
            "text_is_null", "url_is_null", "language_score_is_null", "token_count_is_null",
            "rule_token_lt_50", "rule_text_len_lt_200", "rule_lang_lt_085",
            "rule_token_gt_10000", "rule_url_blocked", "drop_reason_count"
        ]

        append_sample_rows(cleaned_samples, kept[cleaned_cols], CLEANED_SAMPLE_N)
        append_sample_rows(dropped_samples, dropped[dropped_cols], DROPPED_SAMPLE_N)
        append_sample_rows(borderline_samples, borderline[dropped_cols], BORDERLINE_SAMPLE_N)

cleaned_sample_df = pd.concat(cleaned_samples, ignore_index=True) if cleaned_samples else pd.DataFrame()
dropped_sample_df = pd.concat(dropped_samples, ignore_index=True) if dropped_samples else pd.DataFrame()
borderline_sample_df = pd.concat(borderline_samples, ignore_index=True) if borderline_samples else pd.DataFrame()

# trim sample size exactly
cleaned_sample_df = cleaned_sample_df.head(CLEANED_SAMPLE_N)
dropped_sample_df = dropped_sample_df.head(DROPPED_SAMPLE_N)
borderline_sample_df = borderline_sample_df.head(BORDERLINE_SAMPLE_N)

cleaned_sample_path = os.path.join(OUTPUT_DIR, "cleaned_samples.csv")
dropped_sample_path = os.path.join(OUTPUT_DIR, "dropped_samples.csv")
borderline_sample_path = os.path.join(OUTPUT_DIR, "borderline_samples.csv")

cleaned_sample_df.to_csv(cleaned_sample_path, index=False)
dropped_sample_df.to_csv(dropped_sample_path, index=False)
borderline_sample_df.to_csv(borderline_sample_path, index=False)

print("Done.")
print("Total rows scanned:", total_rows)
print("Saved cleaned sample ->", os.path.abspath(cleaned_sample_path))
print("Saved dropped sample ->", os.path.abspath(dropped_sample_path))
print("Saved borderline sample ->", os.path.abspath(borderline_sample_path))


Processing: data/fineweb/004_00018.parquet
Done.
Total rows scanned: 172447
Saved cleaned sample -> /home/jovyan/filter_outputs/cleaned_samples.csv
Saved dropped sample -> /home/jovyan/filter_outputs/dropped_samples.csv
Saved borderline sample -> /home/jovyan/filter_outputs/borderline_samples.csv



## 3. Basic filtering results


In [6]:

dropped_unique_rows_est = None
if not dropped_sample_df.empty:
    pass

rows_dropped_estimate = None  # 这里只是 sample 变量，不代表全量
# 真正的全量保留/删除在下面计算：

rows_dropped_by_any_rule = 0
rows_kept_after_filter = 0

for parquet_path in PARQUET_FILES:
    dataset = ds.dataset(parquet_path, format="parquet")
    scanner = dataset.scanner(columns=columns_to_read, batch_size=BATCH_SIZE)
    for batch in scanner.to_batches():
        df = batch.to_pandas()
        flagged = compute_rule_flags(df)
        rows_dropped_by_any_rule += int(flagged["is_dropped"].sum())
        rows_kept_after_filter += int((~flagged["is_dropped"]).sum())

retention_rate = rows_kept_after_filter / total_rows if total_rows else 0.0
drop_rate = rows_dropped_by_any_rule / total_rows if total_rows else 0.0

basic_results = pd.DataFrame({
    "metric": [
        "total_rows",
        "rows_kept_after_filter",
        "rows_dropped_by_any_rule",
        "retention_rate",
        "drop_rate",
    ],
    "value": [
        total_rows,
        rows_kept_after_filter,
        rows_dropped_by_any_rule,
        retention_rate,
        drop_rate,
    ]
})

display(basic_results)

basic_results.to_csv(os.path.join(OUTPUT_DIR, "basic_filtering_results.csv"), index=False)


,metric,value
0,total_rows,172447.000000
1,rows_kept_after_filter,165738.000000
2,rows_dropped_by_any_rule,6709.000000
3,retention_rate,0.961095
4,drop_rate,0.038905



## 4. Before cleaning vs. after cleaning: core distribution metrics


In [7]:

before_after_summary = pd.DataFrame([
    {"metric": "language_score", "stage": "before", **describe_series(before_lang_values)},
    {"metric": "language_score", "stage": "after", **describe_series(after_lang_values)},
    {"metric": "token_count", "stage": "before", **describe_series(before_token_values)},
    {"metric": "token_count", "stage": "after", **describe_series(after_token_values)},
    {"metric": "text_length", "stage": "before", **describe_series(before_text_len_values)},
    {"metric": "text_length", "stage": "after", **describe_series(after_text_len_values)},
])

display(before_after_summary)
before_after_summary.to_csv(os.path.join(OUTPUT_DIR, "before_after_core_stats.csv"), index=False)


,metric,stage,count,mean,median,min,p25,p75,max
0,language_score,before,172447,0.935297,0.943012,0.650054,0.918842,0.962026,0.998501
1,language_score,after,165738,0.940221,0.944382,0.850014,0.922262,0.962716,0.998501
2,token_count,before,172447,779.813833,487.000000,31.000000,241.000000,916.000000,103654.000000
3,token_count,after,165738,762.924827,500.500000,50.000000,251.000000,928.000000,9970.000000
4,text_length,before,172447,3638.066304,2262.000000,136.000000,1111.000000,4286.000000,481614.000000
5,text_length,after,165738,3574.742920,2332.000000,207.000000,1167.000000,4356.000000,53704.000000


In [8]:

print("=== Top domains before ===")
display(pd.DataFrame(before_domain_counter.most_common(20), columns=["domain", "count"]))

print("=== Top domains after ===")
display(pd.DataFrame(after_domain_counter.most_common(20), columns=["domain", "count"]))

pd.DataFrame(before_domain_counter.most_common(100), columns=["domain", "count"]).to_csv(
    os.path.join(OUTPUT_DIR, "top_domains_before.csv"), index=False
)
pd.DataFrame(after_domain_counter.most_common(100), columns=["domain", "count"]).to_csv(
    os.path.join(OUTPUT_DIR, "top_domains_after.csv"), index=False
)


=== Top domains before ===


,domain,count
0,pubmed.ncbi.nlm.nih.gov,59
1,plandisney.disney.go.com,53
2,telegraph.co.uk,46
3,ibtimes.co.uk,43
4,cbsnews.com,42
5,latimes.com,38
6,sciencedaily.com,38
7,prweb.com,38
8,timesofindia.indiatimes.com,37
9,publications.parliament.uk,33


=== Top domains after ===


,domain,count
0,pubmed.ncbi.nlm.nih.gov,59
1,plandisney.disney.go.com,53
2,telegraph.co.uk,46
3,ibtimes.co.uk,43
4,cbsnews.com,42
5,latimes.com,38
6,sciencedaily.com,38
7,prweb.com,38
8,timesofindia.indiatimes.com,37
9,publications.parliament.uk,33



## 5.  Rule Hit Counts: Which Rules Are Removing Large Amounts of Data



In [9]:

rule_hit_df = pd.DataFrame(rule_hit_counter.items(), columns=["rule", "hit_count"]).sort_values(
    by="hit_count", ascending=False
)
display(rule_hit_df)
rule_hit_df.to_csv(os.path.join(OUTPUT_DIR, "rule_hit_counts.csv"), index=False)


,rule,hit_count
6,language_score < 0.85,6126
7,token_count > 10000,326
8,url path blocked,250
4,token_count < 50,74
5,len(text) < 200,65
0,text is null,0
3,token_count is null,0
1,url is null,0
2,language_score is null,0



## 6. Dirty Data Signal Comparison

Compare here:

* boilerplate hit rate
* HTML residue rate


In [10]:

noise_df = pd.DataFrame([
    {
        "metric": "boilerplate_doc_ratio",
        "before": before_boilerplate_docs / total_rows if total_rows else 0.0,
        "after": after_boilerplate_docs / rows_kept_after_filter if rows_kept_after_filter else 0.0,
    },
    {
        "metric": "html_residue_doc_ratio",
        "before": before_html_docs / total_rows if total_rows else 0.0,
        "after": after_html_docs / rows_kept_after_filter if rows_kept_after_filter else 0.0,
    },
])

display(noise_df)
noise_df.to_csv(os.path.join(OUTPUT_DIR, "noise_signal_comparison.csv"), index=False)


,metric,before,after
0,boilerplate_doc_ratio,0.060558,0.059691
1,html_residue_doc_ratio,0.000510,0.000446



## 7. **Sample Inspection**


### 7.1 Retained Samples（cleaned）
### 7.2 Removed Samples（dropped）
### 7.3 Borderline Samples（borderline: language_score 在 0.85 ~ 0.90）


In [11]:

print("=== Cleaned samples ===")
display(cleaned_sample_df.head(20))

print("=== Dropped samples ===")
display(dropped_sample_df.head(20))

print("=== Borderline samples ===")
display(borderline_sample_df.head(20))


=== Cleaned samples ===


,url,domain,language_score_num,token_count_num,text_length,boilerplate_hits,has_html_residue,text
0,https://www.townsquareapp.co/,townsquareapp.co,0.899284,203,899,0,False,"Welcome to Harrisonburg TownSquare.\nTownSquare brings together all your favorite local shops and makers in one cozy corner of the internet, so you can support your neighbors and find something tr..."
1,https://www.traact.com/blog/the-art-of-effective-board-reporting,traact.com,0.942698,2717,15150,0,False,"The Art of Effective Board Reporting\nSep 5, 2024\nBoard reporting is a cornerstone of corporate governance, serving as the primary channel through which the board of directors gains insight into ..."
2,https://www.tracydixonmindandbodyfit.com/post/2019/08/28/i-wanted-to-be-around-for-my-daughter-when-she-grew-up-i-knew-i-needed-to-make-changes,tracydixonmindandbodyfit.com,0.987144,555,2512,0,False,Belinda's journey and transformation is a real triumph over so much personal stress and pressure. I have a huge amount of respect and admiration for Belinda who has kept going despite all the setb...
3,https://www.transportdirectory.org/items-that-allow-the-use-of-embroidery-in-olathe/,transportdirectory.org,0.976207,469,2141,0,False,Many people purchase plain shirts or other clothing items so they can later add embroidery and designs to them. They want to personalize their own items to make them more their own. Embroidering i...
4,https://www.travelchinaguide.com/package/5-days-in-beijing.htm,travelchinaguide.com,0.946699,1496,6327,0,False,"Beijing Itinerary 5 Days: What to See & How to Plan\nDay 1: Tiananmen Square, Forbidden City, Jingshan Park, Beihai Park, Wangfujing\nYou may go first to Tiananmen Square and Forbidden City, for t..."
5,https://www.travelfortravellers.com/2017/08/5-reasons-visit-hong-kong-taiwan/,travelfortravellers.com,0.907712,1440,6332,0,False,"The enchantment of Hong Kong is undeniable: city wonders, a glimmering harbour, and upbeat energy fill the city. Just over 700 kilometres across the East China Sea in Taiwan lies an otherworldly l..."
6,https://www.travelwithjan.com/america2007yahja,travelwithjan.com,0.903129,768,3317,0,False,"Yahja: ""Jan the Survivor""\nJanuary 5, 2007\nDear Family and Friends,\nHat Co...Sa Qax,\nYes, I am a survivor!\nSo far on this trip to Centro America I have survived the four hour angry-bull-ride d..."
7,https://www.treatmyocd.com/blog/postpartum-intrusive-thoughts,treatmyocd.com,0.958467,2985,14174,0,False,"Having a baby is often described as an exciting time filled with love, joy, and new experiences. However, the reality for many new parents is more complex. Yes, there are beautiful moments, but th..."
8,https://www.trinitylandbuyer.com/,trinitylandbuyer.com,0.953812,921,3940,0,False,Do You Need To Sell Raw Land in Florida and Throughout the USA That’s Just Sitting There Costing You Money?\nTurn that land into CASH now!\nIf you need to sell your property for any reason we can ...
9,https://www.truckandcar.ca/,truckandcar.ca,0.951904,392,1824,0,False,Find Your Next Vehicle\nAsk the Seller for a CARFAX Canada Report\nWhat is a CARFAX Canada report?\n- CARFAX Canada provides comprehensive vehicle history reporting in Canada\nWhat does CARFAX Can...


=== Dropped samples ===


,url,domain,language_score_num,token_count_num,text_length,boilerplate_hits,has_html_residue,text,text_is_null,url_is_null,language_score_is_null,token_count_is_null,rule_token_lt_50,rule_text_len_lt_200,rule_lang_lt_085,rule_token_gt_10000,rule_url_blocked,drop_reason_count
0,https://www.voltpiint.shop/products/blue-sneakers-for-men,voltpiint.shop,0.834038,114,508,0,False,"The LOUIS STITCH PLAY Sneakers take their inspiration from the dynamic rhythm of playful urban life. Heartcrafted with utmost perfection, these sneakers feature comfortable yet stylish and modern ...",False,False,False,False,False,False,True,False,False,1
1,https://www.willowshoes.com/collections/sneakers/products/frankie4-mim-iv-white-suede-sneakers-for-large-feet,willowshoes.com,0.825000,115,490,0,False,"Regular price $299.00Rich textures and a raised sole perfectly balances Mim’s elevated silhouette, in classic White Suede. Featuring Frankie4's renowned footbed technology, dual-density moulded so...",False,False,False,False,False,False,True,False,False,1
2,https://www.yamahawaverunners.com/waverunner/series/performance/GP-SVHO/,yamahawaverunners.com,0.803809,577,2569,0,False,"Supercharged 4-cylinder, 4-stroke, Super Vortex High Output Yamaha Marine Engine\nFuel Capacity (gal)\nFuel Capacity (L)\nOil Capacity per Engine (L)\nReverse Assist Mode\nFactory Installed Yamaha...",False,False,False,False,False,False,True,False,False,1
3,https://ads.tiktok.com/business/en-GB/products/ads,ads.tiktok.com,0.840870,67,310,0,False,"Campaign solutions designed to boost visibility, engagement and reach.\nStart a conversation directly in your feed\nTell your brand story in a true-to-TikTok way with this sound-on, vertical video...",False,False,False,False,False,False,True,False,False,1
4,https://agrarzone.com/dental-care-dog,agrarzone.com,0.765003,807,3227,0,False,"Dog dental care: For healthy and happy dogs\nGood dental care is just as important for dogs as it is for us humans. Clean and healthy teeth not only contribute to fresh breath, but also prevent de...",False,False,False,False,False,False,True,False,False,1
5,https://aieris.art/featured/night-windows-eris-and-ai.html?product=poster,aieris.art,0.797187,112,517,0,False,Night Windows poster by Eris And AI. Our posters are produced on acid-free papers using archival inks to guarantee that they last a lifetime without fading or loss of color. All posters include a ...,False,False,False,False,False,False,True,False,False,1
6,https://app.jove.com/author/48522/syed-benazir-alam,app.jove.com,0.655934,354,1202,0,False,EN - English\nCN - 中文\nDE - Deutsch\nES - Español\nKR - 한국어\nIT - Italiano\nFR - Français\nPT - Português\nTR - Türkçe\nJA - 日本語\nPL - Polski\nRU - Русский\nHE - עִברִית\nAR - العربية\nNanotechnol...,False,False,False,False,False,False,True,False,False,1
7,https://ariabookmarks.com/story4622443/social-media-content-marketing-virtual-assistant-services,ariabookmarks.com,0.702255,203,542,0,False,"129 days ago\nSubmit a Comment\nHTML is disabled\nWho Upvoted this Story\nSculpting Your Physique: Killer Workouts for Ma...\nMédicos en el recorrido del maratón: esenciales...\nDetails, Fiction a...",False,False,False,False,False,False,True,False,False,1
8,https://bestlasercutterforsmallbusiness.com/product-category/whats-a-good-diode-laser-for-engraving-tumblers/,bestlasercutterforsmallbusiness.com,0.831828,224,1045,0,False,"Discover the ultimate precision and versatility with the Hawk P2S & P2✓Laser Engraver For Cups✓Good Laser Cutter. Designed for both hobbyists and professionals, this advanced laser engraver offers...",False,False,False,False,False,False,True,False,False,1
9,https://bookmarkvids.com/story19448249/new-past-30-days,bookmarkvids.com,0.704349,211,717,0,False,258 days ago\nSubmit a Comment\nHTML is disabled\nWho Upvoted this Story\nA Review Of hire bodyguards in London\nKaufen Sie einen copyright Options\nGüngören'deki Müşteri Odaklı Hizmetler\nProcure...,False,False,False,False,False,False,True,False

=== Borderline samples ===


,url,domain,language_score_num,token_count_num,text_length,boilerplate_hits,has_html_residue,text,text_is_null,url_is_null,language_score_is_null,token_count_is_null,rule_token_lt_50,rule_text_len_lt_200,rule_lang_lt_085,rule_token_gt_10000,rule_url_blocked,drop_reason_count
0,https://www.townsquareapp.co/,townsquareapp.co,0.899284,203,899,0,False,"Welcome to Harrisonburg TownSquare.\nTownSquare brings together all your favorite local shops and makers in one cozy corner of the internet, so you can support your neighbors and find something tr...",False,False,False,False,False,False,False,False,False,0
1,https://www.unitedstatesofganja.com/2025/01/22/exploring-the-thrill-of-online-pokies-in-australia-top-casinos-and-games/,unitedstatesofganja.com,0.881102,1533,7533,0,False,Exploring the Thrill of Online Pokies in Australia Top Casinos and Games\nExperience the thrill of pokies online like never before! Whether you’re searching for real pokies or the best online casi...,False,False,False,False,False,False,False,False,False,0
2,https://www.vancouverclub.ca/posts/hotel-rooms-reservation-guide-website,vancouverclub.ca,0.887821,534,2522,0,False,"The following online reservations are available through your member website: Club events, à la carte dining and hotel rooms.\nYour website also includes information for booking additional Club ser...",False,False,False,False,False,False,False,False,False,0
3,https://www.vetmeduni.ac.at/en/fundraising/your-donation-for-our-animal-hospital,vetmeduni.ac.at,0.887206,97,460,0,False,"Your donation for our animal hospital\nYour donation goes directly to the treatment of the patient's animals, the high-quality clinic equipment (such as the most modern devices for tumor treatment...",False,False,False,False,False,False,False,False,False,0
4,https://www.vintagewinemerchants.com/products/2019-kathryn-kennedy-cabernet-sauvignon-estate,vintagewinemerchants.com,0.878465,189,837,0,False,"93 Wine Enthusiast - The region's forest-like qualities show on the nose of this bottling, where hints of sandalwood, cigar box, oregano and thyme add to the dried red-fruit aromas. The palate off...",False,False,False,False,False,False,False,False,False,0
5,https://www.visitarizona.com/events/lake-powell-sxs-extravaganza/,visitarizona.com,0.896914,373,1642,0,False,Lake Powell SXS Extravaganza\nOct 10th – Oct 12th\n- Time4:00 PM to 5:00 PM\nWelcome to the Lake Powell SXS Extravaganza!\nGet ready for a heart-pounding adventure set against the stunning backdro...,False,False,False,False,False,False,False,False,False,0
6,https://www.vitarsis.com/?v=5c2a35ef64b8,vitarsis.com,0.870176,249,1266,0,False,"Unlock Your Inner Potential\nReady for meaningful change? Vitarsis equips you with unique tools and focused approaches to help you elevate your mindset, enhance your focus, and build the life you ...",False,False,False,False,False,False,False,False,False,0
7,https://www.vtmarkets.com/notification/v2023030801/,vtmarkets.com,0.892725,129,559,0,False,"Notification of Server Upgrade – Jun 12 ,2025\nDear Client, As part of our commitment to provide the most reliable service to our clients, there will be maintenance …\nThe adjustment of DST will b...",False,False,False,False,False,False,False,False,False,0
8,https://www.willerby.com/parks/south-west/somerset/weston-super-mare,willerby.com,0.866001,268,1132,0,False,"Steeped with Victorian history and dominated by a glorious beach\nFind out more about this Victorian splendour at the Weston Museum, or explore 40 acres of rolling Somerset countryside Puxton Park...",False,False,False,False,False,False,False,False,False,0
9,https://www.windowsbasics.com/2021/10/how-to-change-font-size-on-windows-11.html,windowsbasics.com,0.877272,327,1512,0,False,"Font size has a huge impact on user experience on any operating system platform, and Windows 11 is no exception. To enlarge or reduce the displayed text size on Windows 11, you just need to follow...",False,False,False,False,False,False,False,False


## 8. Short summaries of findings


In [12]:

def safe_stat(df, metric, stage, col):
    row = df[(df["metric"] == metric) & (df["stage"] == stage)]
    if len(row) == 0 or col not in row.columns:
        return None
    v = row.iloc[0][col]
    return v if pd.notna(v) else None

lang_before_med = safe_stat(before_after_summary, "language_score", "before", "median")
lang_after_med = safe_stat(before_after_summary, "language_score", "after", "median")

token_before_med = safe_stat(before_after_summary, "token_count", "before", "median")
token_after_med = safe_stat(before_after_summary, "token_count", "after", "median")

text_before_med = safe_stat(before_after_summary, "text_length", "before", "median")
text_after_med = safe_stat(before_after_summary, "text_length", "after", "median")

summary_lines = [
    f"Total rows scanned: {total_rows}",
    f"Rows kept after filtering: {rows_kept_after_filter} ({retention_rate:.2%})",
    f"Rows dropped by at least one rule: {rows_dropped_by_any_rule} ({drop_rate:.2%})",
]

if lang_before_med is not None and lang_after_med is not None:
    summary_lines.append(
        f"Median language_score changed from {lang_before_med:.4f} to {lang_after_med:.4f}."
    )

if token_before_med is not None and token_after_med is not None:
    summary_lines.append(
        f"Median token_count changed from {token_before_med:.2f} to {token_after_med:.2f}."
    )

if text_before_med is not None and text_after_med is not None:
    summary_lines.append(
        f"Median text length changed from {text_before_med:.2f} to {text_after_med:.2f}."
    )

summary_lines.append(
    f"Boilerplate ratio changed from {noise_df.loc[noise_df['metric']=='boilerplate_doc_ratio', 'before'].iloc[0]:.2%} "
    f"to {noise_df.loc[noise_df['metric']=='boilerplate_doc_ratio', 'after'].iloc[0]:.2%}."
)
summary_lines.append(
    f"HTML residue ratio changed from {noise_df.loc[noise_df['metric']=='html_residue_doc_ratio', 'before'].iloc[0]:.2%} "
    f"to {noise_df.loc[noise_df['metric']=='html_residue_doc_ratio', 'after'].iloc[0]:.2%}."
)

summary_text = "\n".join("- " + line for line in summary_lines)
print(summary_text)

with open(os.path.join(OUTPUT_DIR, "short_summary.txt"), "w", encoding="utf-8") as f:
    f.write(summary_text)


- Total rows scanned: 172447
- Rows kept after filtering: 165738 (96.11%)
- Rows dropped by at least one rule: 6709 (3.89%)
- Median language_score changed from 0.9430 to 0.9444.
- Median token_count changed from 487.00 to 500.50.
- Median text length changed from 2262.00 to 2332.00.
- Boilerplate ratio changed from 6.06% to 5.97%.
- HTML residue ratio changed from 0.05% to 0.04%.



## 9. What should be manually reviewed first afterward

After the run, focus on these files:

- `filter_outputs/cleaned_samples.csv`
- `filter_outputs/dropped_samples.csv`
- `filter_outputs/borderline_samples.csv`
- `filter_outputs/rule_hit_counts.csv`
- `filter_outputs/noise_signal_comparison.csv`
- `filter_outputs/before_after_core_stats.csv`


Especially check:

1. Whether most of the dropped samples really deserved to be removed
2. Among the borderline samples with `language_score` in `0.85 ~ 0.90`, how many are actually good text
3. Whether the proportions of boilerplate and residual HTML have actually decreased



In [4]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.
